# Colab Full Notebook Runner

Run this notebook on a Google Colab GPU runtime, including through the VS Code Colab extension. It clones or pulls the project into Colab, installs Colab dependencies, runs the corrected questionnaire event-episode pipeline, and can run Tier-2 deep-learning reproduction.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/bahaa1515/EECE-693-project.git"
REPO_DIR = Path("/content/EECE-693-project")

if REPO_DIR.exists():
    %cd /content/EECE-693-project
    !git pull --ff-only
else:
    %cd /content
    !git clone {REPO_URL} EECE-693-project
    %cd /content/EECE-693-project


In [ ]:
!pip install -q -r requirements-colab.txt


In [ ]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


In [ ]:
# Optional Google Drive setup for the full AAMOS dataset and outputs.
# Upload/copy the dataset folder to Drive, then adjust these paths if needed.
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped:", exc)

candidate_data_dir = Path(os.environ.get("AAMOS_DATA_DIR", "/content/drive/MyDrive/AAMOS/dataset"))
candidate_output_dir = Path(os.environ.get("AAMOS_OUTPUT_DIR", "/content/drive/MyDrive/AAMOS/outputs"))

if candidate_data_dir.exists():
    os.environ["AAMOS_DATA_DIR"] = str(candidate_data_dir)
    os.environ["AAMOS_OUTPUT_DIR"] = str(candidate_output_dir)
    print("AAMOS_DATA_DIR=", os.environ["AAMOS_DATA_DIR"])
    print("AAMOS_OUTPUT_DIR=", os.environ["AAMOS_OUTPUT_DIR"])
else:
    print("Set AAMOS_DATA_DIR to the Drive folder containing the full AAMOS CSV files before running the corrected pipeline.")


In [ ]:
# Executes the canonical Tier-2 + Tier-3 pipeline (5 seeds, 20 epochs).
!python scripts/run_full_pipeline.py --full


In [ ]:
# Corrected questionnaire-only labels and full sensitivity sample counts.
!python scripts/run_questionnaire_event_pipeline.py --skip-features


In [ ]:
# Full sensor-feature + tabular-model smoke experiment.
!python scripts/run_questionnaire_event_pipeline.py --thresholds 3 --input-lengths 7 --washouts 7


In [ ]:
!python -m src.deep_learning --epochs 20 --batch-size 64 --patience 5


In [ ]:
import pandas as pd
pd.read_csv("outputs/tables/deep_learning_reproduced_results.csv")


In [ ]:
comparison_path = "outputs/tables/deep_learning_report_comparison.csv"
try:
    display(pd.read_csv(comparison_path))
except FileNotFoundError:
    print(f"No comparison file found at {comparison_path}; reproduced results still saved.")


In [ ]:
from pathlib import Path

for result_path in [
    "outputs/tables/baseline_model_results.csv",
    "outputs/tables/cv_summary_tier1.csv",
    "outputs/tables/deep_learning_reproduced_results.csv",
    "outputs/tables/deep_learning_report_comparison.csv",
]:
    p = Path(result_path)
    status = "OK" if p.exists() else "missing"
    print(f"{result_path}: {status}")
